# 🎬 Visualizador de Fadiga em Vídeo

Reproduz seu vídeo em tempo real mostrando se a pessoa está **Alerta** ou **Sonolenta**.

## Como usar:
1. Execute as células em ordem
2. Escolha o modelo (Normal ou Optuna)
3. Digite o caminho do seu vídeo
4. Assista ao vídeo com as predições

**Controles**: ESPAÇO=pausar/continuar | ESC=sair

In [ ]:
import numpy as np
import cv2
import mediapipe as mp
import time
from collections import deque
from pathlib import Path
import sys
import json

print("Bibliotecas carregadas")

In [ ]:
# Seleção do Modelo
def selecionar_modelo():
    print("=== SELEÇÃO DO MODELO ===")
    modelos = []
    
    # Modelo Normal
    if Path("modelos_xgb/modelo_xgb.joblib").exists():
        try:
            with open("modelos_xgb/info_classes.json") as f:
                info = json.load(f)
            acc = info.get('accuracy', 0) * 100
            print(f"1. Normal - {acc:.1f}% acurácia")
            modelos.append(('normal', 'modelos_xgb', acc))
        except:
            print("1. Normal - disponível")
            modelos.append(('normal', 'modelos_xgb', 0))
    
    # Modelo Optuna  
    if Path("modelos_xgb_optuna/modelo_xgb.joblib").exists():
        try:
            with open("modelos_xgb_optuna/info_classes.json") as f:
                info = json.load(f)
            acc = info.get('accuracy', 0) * 100
            auc = info.get('auc', 0)
            print(f"2. Optuna - {acc:.1f}% acurácia | AUC: {auc:.3f}")
            modelos.append(('optuna', 'modelos_xgb_optuna', acc))
        except:
            print("2. Optuna - disponível")
            modelos.append(('optuna', 'modelos_xgb_optuna', 0))
    
    if not modelos:
        raise Exception("Nenhum modelo encontrado! Execute os notebooks de treino primeiro.")
    
    # Auto-seleciona melhor ou pergunta
    if len(modelos) == 1:
        escolhido = modelos[0]
        print(f"Auto-selecionado: {escolhido[0].title()}")
    else:
        melhor = max(modelos, key=lambda x: x[2])
        print(f"Recomendado: {melhor[0].title()}")
        
        escolha = input("\nEscolha (1/2/Enter=melhor): ").strip()
        if escolha == "1":
            escolhido = next(m for m in modelos if m[0] == 'normal')
        elif escolha == "2":
            escolhido = next(m for m in modelos if m[0] == 'optuna')
        else:
            escolhido = melhor
    
    print(f"Usando: {escolhido[0].title()} - {escolhido[2]:.1f}%")
    return escolhido[0], escolhido[1]

tipo_modelo, modelo_dir = selecionar_modelo()

In [ ]:
# Inicialização
print("Inicializando sistemas...")

# MediaPipe
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Modelo
sys.path.append(modelo_dir)
from pipeline import PipelineFadiga

pipeline = PipelineFadiga(modelo_dir)

print("Sistemas prontos!")

In [ ]:
# Detector Simples para Vídeo
class DetectorVideoSimples:
    def __init__(self, pipeline):
        self.pipeline = pipeline
        self.buffer = deque(maxlen=90)
        self.ear_history = deque(maxlen=200)
        
        # Pontos MediaPipe
        self.olho_direito = [33, 160, 158, 133, 153, 144]
        self.olho_esquerdo = [362, 385, 387, 263, 373, 380]
        self.boca = [13, 14, 78, 308, 0, 17]
        
        # Estado atual
        self.predicao = "Analisando..."
        self.confianca = 0.0
        self.ear_atual = 0.0
        self.mar_atual = 0.0
    
    def distancia(self, p1, p2):
        return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
    
    def calcular_ear(self, pontos_olho, landmarks, w, h):
        try:
            pontos = [[landmarks[i].x * w, landmarks[i].y * h] for i in pontos_olho]
            if len(pontos) >= 6:
                d1 = self.distancia(pontos[1], pontos[5])
                d2 = self.distancia(pontos[2], pontos[4])
                d3 = self.distancia(pontos[0], pontos[3])
                if d3 > 0:
                    return max(0.0, min(1.0, (d1 + d2) / (2.0 * d3)))
            return 0.0
        except:
            return 0.0
    
    def calcular_mar(self, landmarks, w, h):
        try:
            pontos = [[landmarks[i].x * w, landmarks[i].y * h] for i in self.boca]
            if len(pontos) >= 6:
                d_vert = self.distancia(pontos[0], pontos[1]) + self.distancia(pontos[4], pontos[5]) * 0.5
                d_horiz = self.distancia(pontos[2], pontos[3])
                if d_horiz > 0:
                    return max(0.0, min(3.0, d_vert / d_horiz))
            return 0.0
        except:
            return 0.0
    
    def processar_frame(self, frame):
        h, w = frame.shape[:2]
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)
        
        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            
            # Calcula métricas
            ear_d = self.calcular_ear(self.olho_direito, landmarks, w, h)
            ear_e = self.calcular_ear(self.olho_esquerdo, landmarks, w, h)
            self.ear_atual = (ear_d + ear_e) / 2.0
            self.mar_atual = self.calcular_mar(landmarks, w, h)
            
            self.ear_history.append(self.ear_atual)
            
            # PERCLOS simples
            if len(self.ear_history) >= 60:
                window = list(self.ear_history)[-60:]
                frames_fechados = sum(1 for e in window if e < 0.25)
                perclos = (frames_fechados / len(window)) * 100
            else:
                perclos = 0.0
            
            # Features simples
            features = [perclos, self.mar_atual, 0.0, 0.0]
            self.buffer.append(features)
            
            return True
        return False
    
    def predizer(self):
        if len(self.buffer) == 90:
            sequence = np.array(list(self.buffer))
            pred, probs, classe = self.pipeline.predict_sequence(sequence)
            self.predicao = classe
            self.confianca = max(probs.values())
            return True
        return False
    
    def desenhar_status(self, frame):
        h, w = frame.shape[:2]
        
        # Fundo para o status
        overlay = frame.copy()
        cv2.rectangle(overlay, (20, 20), (400, 120), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.8, frame, 0.2, 0, frame)
        
        # Status principal
        cor = (0, 255, 0) if self.predicao == "Alerta" else (0, 0, 255) if self.predicao == "Sonolento" else (255, 255, 255)
        cv2.putText(frame, f"STATUS: {self.predicao}", (30, 60), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, cor, 3)
        
        # Confiança
        if self.confianca > 0:
            cv2.putText(frame, f"Confianca: {self.confianca:.0%}", (30, 95), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Buffer status
        buffer_text = f"Buffer: {len(self.buffer)}/90"
        cv2.putText(frame, buffer_text, (w-200, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Controles
        cv2.putText(frame, "ESPACO=pausar | ESC=sair", (20, h-20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        return frame

detector = DetectorVideoSimples(pipeline)
print("Detector de vídeo criado!")

In [ ]:
# Configuração do Vídeo
def obter_caminho_video():
    print("CONFIGURAÇÃO DO VÍDEO")
    print("Formatos: MP4, AVI, MOV, MKV, WMV")
    print()
    
    while True:
        caminho = input("Digite o caminho do vídeo: ").strip().strip('"\'')
        
        if not caminho:
            print("Digite um caminho válido.")
            continue
            
        arquivo = Path(caminho)
        
        if not arquivo.exists():
            print(f"Arquivo não encontrado: {arquivo}")
            continue
            
        # Testa se consegue abrir
        cap = cv2.VideoCapture(str(arquivo))
        if not cap.isOpened():
            print("Não foi possível abrir o vídeo.")
            cap.release()
            continue
            
        # Info do vídeo
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duracao = frames / fps if fps > 0 else 0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        cap.release()
        
        print(f"Vídeo carregado:")
        print(f"   {arquivo.name}")
        print(f"   {width}x{height}")
        print(f"   {duracao:.1f}s ({fps:.1f} FPS)")
        
        return str(arquivo)

caminho_video = obter_caminho_video()

In [ ]:
# REPRODUZIR VÍDEO COM DETECÇÃO

def reproduzir_video_com_deteccao(caminho_video):
    print("INICIANDO REPRODUÇÃO...")
    print("Controles: ESPAÇO=pausar/continuar | ESC=sair")
    print()
    
    cap = cv2.VideoCapture(caminho_video)
    if not cap.isOpened():
        print("Erro ao abrir vídeo")
        return
    
    # Propriedades
    fps = cap.get(cv2.CAP_PROP_FPS)
    delay = int(1000 / fps) if fps > 0 else 33
    
    # Reset do detector
    global detector
    detector = DetectorVideoSimples(pipeline)
    
    cv2.namedWindow('Detecção de Fadiga - Vídeo', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Detecção de Fadiga - Vídeo', 1000, 700)
    
    pausado = False
    frame_count = 0
    
    print(" Reproduzindo vídeo...")
    
    try:
        while True:
            if not pausado:
                ret, frame = cap.read()
                if not ret:
                    print("\nFim do vídeo")
                    break
                
                frame_count += 1
                
                # Processa detecção
                face_detectada = detector.processar_frame(frame)
                if face_detectada:
                    detector.predizer()
                
                # Desenha interface
                frame_final = detector.desenhar_status(frame)
                
                # Status da face no canto
                h, w = frame_final.shape[:2]
                status_face = "Face OK" if face_detectada else "Sem Face"
                cor_face = (0, 255, 0) if face_detectada else (0, 0, 255)
                cv2.putText(frame_final, status_face, (w-200, 60), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor_face, 2)
            
            # Mostra frame
            cv2.imshow('Detecção de Fadiga - Vídeo', frame_final)
            
            # Controles
            key = cv2.waitKey(delay if not pausado else 1) & 0xFF
            
            if key == 27:
                print("\nReprodução interrompida")
                break
            elif key == 32:
                pausado = not pausado
                status = "PAUSADO" if pausado else "REPRODUZINDO"
                print(f"{status}")
            
            # Verifica se janela foi fechada
            if cv2.getWindowProperty('Detecção de Fadiga - Vídeo', cv2.WND_PROP_VISIBLE) < 1:
                break
                
    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário")
    
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print(f"\nReprodução finalizada! Frames processados: {frame_count}")

# Inicia reprodução
reproduzir_video_com_deteccao(caminho_video)